# Regional equation-library comparison

Interpretable one-frame maps are fitted separately within Awake, NREM, and REM using chronological episode splits. The primary analysis uses causal 16 s high-pass residuals so that equation terms are not rewarded merely for reproducing episode baselines.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig

ROOT = Path('..').resolve()
RUN = ROOT / 'logs/scripts/regional_equation_library/initial'
OUT = ROOT / 'figures/261027区域方程库比较'
OUT.mkdir(parents=True, exist_ok=True)
summary = pd.read_csv(RUN / 'model_summary.csv')
rollout = pd.read_csv(RUN / 'rollout_metrics.csv')
summary.sort_values(['representation', 'state', 'dx_r2_flat'], ascending=[True, True, False])

,representation,state,model,alpha,validation_mse,level_r2_flat,dx_r2_flat,dx_r2_region_median,dx_corr_flat,n_test_targets,rollout_r2_at_4s,rollout_corr_at_4s,bounded_fraction_at_4s
11,causal_highpass,Awake,full_saturating,1000.00,0.107912,0.885224,0.093720,0.083806,0.308798,5129,-0.132387,0.108423,1.0
10,causal_highpass,Awake,full_linear,100.00,0.109085,0.883479,0.079939,0.070485,0.299447,5129,-0.008628,0.150597,1.0
9,causal_highpass,Awake,global_nonlinear,1000.00,0.114183,0.879108,0.045425,0.040763,0.214187,5129,-0.151128,0.080237,1.0
8,causal_highpass,Awake,self_cubic,1000.00,0.115534,0.877680,0.034152,0.025296,0.188544,5129,-0.098231,0.095401,1.0
7,causal_highpass,Awake,self_linear,1000.00,0.116119,0.877576,0.033330,0.026719,0.188387,5129,-0.055363,0.082363,1.0
6,causal_highpass,Awake,persistence,0.00,0.124230,0.873354,-0.000004,-0.000006,NaN,5129,-0.977390,0.082940,1.0
23,causal_highpass,NREM,full_saturating,100.00,0.171402,0.765395,0.177439,0.124424,0.421627,862,0.061456,0.268843,1.0
22,causal_highpass,NREM,full_linear,100.00,0.171819,0.764813,0.175396,0.129180,0.418944,862,0.024162,0.183684,1.0
21,causal_highpass,NREM,global_nonlinear,0.01,0.188080,0.736950,0.077704,0.056798,0.279586,862,0.035047,0.192292,1.0
20,causal_highpass,NREM,self_cubic,100.00,0.191992,0.733071,0.064105,0.054458,0.256599,862,0.000176,0.085794,1.0


In [2]:
models = ['persistence', 'self_linear', 'self_cubic', 'global_nonlinear', 'full_linear', 'full_saturating']
labels = ['zero dx', 'self linear', 'self cubic', 'global nonlinear', 'full linear', 'full saturating']
colors = {'Awake': '#cc5544', 'NREM': '#4477aa', 'REM': '#aa55aa'}
part = summary[summary.representation == 'causal_highpass']
fig, axes = get_fig(RN=1, CN=2, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(axes)
ax = next(ax_loader)
x = np.arange(len(models))
for offset, state in zip([-.22, 0, .22], ['Awake', 'NREM', 'REM']):
    lookup = part[part.state == state].set_index('model')
    ax.bar(x + offset, lookup.loc[models].dx_r2_flat, width=.21, color=colors[state], label=state)
ax.axhline(0, color='0.25', lw=.8)
ax.set_xticks(x, labels, rotation=45, ha='right')
ax.set(ylabel=r'Held-out $\Delta x$ $R^2$', title='One-frame increment prediction')
ax.legend(frameon=False)
ax = next(ax_loader)
for state, linestyle in [('Awake', '-'), ('NREM', '--')]:
    for model, alpha in [('full_linear', 1.0), ('full_saturating', .55)]:
        curve = rollout[(rollout.representation == 'causal_highpass') & (rollout.state == state) & (rollout.model == model)]
        ax.plot(curve.seconds, curve.r2_flat, color=colors[state], ls=linestyle if model == 'full_linear' else ':', alpha=alpha, label=f'{state}, {model.replace("full_", "")}')
ax.axhline(0, color='0.25', lw=.8)
ax.set(xlim=(0, 8), ylim=(-.5, 1), xlabel='Free-rollout horizon (s)', ylabel=r'Rollout $R^2$', title='Autonomous evolution')
ax.legend(frameon=False, fontsize=9)
fig.savefig(OUT / 'equation_library_performance.png')
fig.savefig(OUT / 'equation_library_performance.svg')
fig.savefig(OUT / 'equation_library_performance.pdf')
plt.show()

## Reading the result

On causal high-pass residuals, full regional coupling is the dominant reproducible addition: full-linear increment R² is about 0.080 in Awake and 0.175 in NREM, versus 0.033 and 0.061 for self-linear equations. Local cubic terms add only about 0.001 and 0.003, respectively.

The saturating network raises one-step increment R² to about 0.094 in Awake but worsens its 4 s autonomous rollout. In NREM its one-step advantage over full linear is only about 0.002, while its 4 s rollout R² improves from about 0.024 to 0.061. This makes state-specific saturation in NREM a candidate for replication, not yet an accepted mechanism. REM has only 248 held-out targets from one episode and should not drive model selection.

Raw-level results remain dominated by persistence. The causal-residual results therefore support a compact working equation consisting first of a state-specific full/low-rank linear regional operator, with nonlinear saturation added only where it improves both episode-held-out increments and autonomous evolution.